In [27]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, OrdinalEncoder, OneHotEncoder
)
from sklearn.impute import SimpleImputer


In [28]:
df = pd.read_csv("../data/raw/Loan_Default.csv")

In [29]:
df.drop("ID", axis=1, inplace=True)

In [30]:
df.shape

(148670, 33)

## Drop columns that found from target leakage audit

In [31]:
df.drop(["rate_of_interest", "Interest_rate_spread", "Upfront_charges", "Neg_ammortization", "loan_limit", "interest_only", "lump_sum_payment"], axis=1, inplace=True)

In [32]:
df.shape

(148670, 26)

In [33]:
df["Security_Type"] = df["Security_Type"].replace("Indriect", "Indirect")

In [34]:
df.head()

,year,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,loan_amount,term,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,2019,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,116500,360.0,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,2019,Male,nopre,type2,p1,l1,nopc,b/c,206500,360.0,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,2019,Male,pre,type1,p1,l1,nopc,nob/c,406500,360.0,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,2019,Male,nopre,type1,p4,l1,nopc,nob/c,456500,360.0,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,2019,Joint,pre,type1,p1,l1,nopc,nob/c,696500,360.0,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


### Fill missing values of `property_value` and calculate LTV value again using that values.

In [35]:
df["property_value"] = df["property_value"].fillna(df["property_value"].median())

df["LTV"] = df["loan_amount"] / df["property_value"]

## Encode Features

In [36]:
ordinal_cols = ["age", "total_units"]
ordinal_categories = [
    ["<25", "25-34", "35-44", "45-54", "55-64", "65-74", ">74"],
    ["1U", "2U", "3U", "4U"]
]


binary_cols = [ "approv_in_adv", "Credit_Worthiness", "open_credit", "business_or_commercial", "construction_type", "Secured_by", "co-applicant_credit_type", "submission_of_application", "Security_Type"]

nominal_cols = ["Gender", "loan_type", "loan_purpose", "occupancy_type", "credit_type", "Region"]

numeric_cols = ["year", "loan_amount","term", "property_value", "income", "Credit_Score", "LTV", "dtir1"]

In [37]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [38]:
onehotencode_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            drop='first'
            ))
    ])

In [39]:
ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

In [40]:
binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories='auto'
    ))
])

In [41]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numeric_cols),
        ("ord", ordinal_pipeline, ordinal_cols),
        ("bin", binary_pipeline, binary_cols),
        ("nom", onehotencode_pipeline, nominal_cols),
    ]
)

In [42]:
X = df.drop('Status', axis=1)
y = df["Status"]

In [43]:
X.columns

Index(['year', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose',
       'Credit_Worthiness', 'open_credit', 'business_or_commercial',
       'loan_amount', 'term', 'property_value', 'construction_type',
       'occupancy_type', 'Secured_by', 'total_units', 'income', 'credit_type',
       'Credit_Score', 'co-applicant_credit_type', 'age',
       'submission_of_application', 'LTV', 'Region', 'Security_Type', 'dtir1'],
      dtype='object')

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y , test_size=0.2, random_state=42, stratify=y
)

In [45]:
print(y_train[:5])
print(type(y_train))
print(y_train.shape)

141039    0
121276    0
11214     1
129659    0
13370     0
Name: Status, dtype: int64
<class 'pandas.core.series.Series'>
(118936,)


In [51]:
y_train.value_counts()

Status
0    89625
1    29311
Name: count, dtype: int64

In [46]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [47]:
X_train_transformed.shape

(118936, 35)

## Save Preprocessor

In [48]:
import os
import joblib

os.makedirs("../artifacts", exist_ok=True)

In [49]:
joblib.dump(preprocessor, "../artifacts/preprocessor.pkl")

['../artifacts/preprocessor.pkl']

In [50]:
# Save train/test splits so training notebook loads same splits
import numpy as np
np.save("../artifacts/X_train.npy", X_train_transformed)
np.save("../artifacts/X_test.npy", X_test_transformed)
np.save("../artifacts/y_train.npy", y_train.values)
np.save("../artifacts/y_test.npy", y_test.values)

print("Artifacts saved.")

Artifacts saved.
